---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 11 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen "Kopi Nusantara" (skenario yang sama dari Pertemuan 7 & 9) ingin membangun **dashboard pemantauan pesanan real-time** untuk mengetahui performa tiap cabang setiap saat, bukan hanya laporan bulanan seperti sebelumnya. Anda ditugaskan membangun purwarupa (*prototype*) pipeline streaming yang mensimulasikan kebutuhan ini.

### Instruksi Pengerjaan

Buat notebook baru **`Tugas10_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession` (jangan lupa `spark.conf.set("spark.sql.shuffle.partitions", "4")`), lalu kerjakan bagian **A sampai E** berikut:

---

**A. Menyiapkan Streaming Source** *(bobot 15%)*

Definisikan skema eksplisit (`StructType`) untuk data pesanan dengan kolom: `pesanan_id` (string), `nama_cabang` (string), `nama_menu` (string), `qty` (integer), `harga_satuan` (integer), `waktu_pesan` (timestamp). Buat folder simulasi streaming baru bernama `tugas10_stream`, lalu definisikan `readStream` dari folder tersebut.

**B. Agregasi per Cabang** *(bobot 20%)*

Tambahkan kolom turunan `total_harga` (`qty x harga_satuan`) pada `df_stream`, lalu buat agregasi `groupBy("nama_cabang")` yang menghitung total `total_harga` dan jumlah pesanan. Jalankan sebagai streaming query dengan sink `memory`, `queryName="agregasi_cabang"`.

**C. Agregasi Berbasis Window Waktu** *(bobot 20%)*

Buat streaming query **kedua** (definisikan ulang `readStream` dari folder yang sama, ini diperbolehkan — satu sumber data dapat dibaca oleh lebih dari satu streaming query sekaligus) yang mengelompokkan total `total_harga` berdasarkan **window waktu 2 menit** dari kolom `waktu_pesan`. Gunakan sink `memory`, `queryName="agregasi_waktu"`.

**D. Menguji dengan Data Simulasi** *(bobot 25%)*

Kirim **minimal 3 batch** data (masing-masing batch berupa satu berkas CSV baru dengan jeda `time.sleep()` di antaranya) ke folder `tugas10_stream`, dengan kombinasi `nama_cabang` yang bervariasi. Setelah setiap batch, tampilkan hasil `agregasi_cabang` untuk membuktikan angka totalnya terus terakumulasi dengan benar.

**E. Sink Berbasis Berkas (Tantangan)** *(bobot 20%)*

Selain `memory` sink, Structured Streaming juga mendukung menulis hasil **DataFrame mentah (bukan hasil agregasi)** ke berkas secara berkelanjutan. Buat **streaming query ketiga** yang membaca dari `df_stream`, lalu menuliskannya (tanpa agregasi apa pun, cukup data mentahnya) ke folder lokal `tugas10_output` dalam format Parquet, menggunakan:
```python
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()
```
Kirim satu batch data baru, tunggu beberapa detik, lalu buktikan berkas Parquet-nya benar-benar tersimpan (`!ls tugas10_output`) dan dapat dibaca kembali sebagai DataFrame batch biasa menggunakan `spark.read.parquet()`.

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas10_[NPM]_[Nama Lengkap].ipynb` melalui asisten praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan seluruh streaming query dihentikan (`query.stop()`) di akhir notebook sebelum dikumpulkan.
- Seluruh cell wajib sudah dijalankan (**Run All**) — perhatikan penggunaan `time.sleep()` yang cukup agar output streaming sempat terisi sebelum notebook mencoba menampilkannya.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Streaming Source | Skema & `readStream` didefinisikan dengan benar | 15% |
| B. Agregasi per Cabang | Kolom turunan & agregasi `groupBy` benar, query berjalan | 20% |
| C. Window Waktu | Agregasi window 2 menit terdefinisi & berjalan dengan benar | 20% |
| D. Pengujian Multi-Batch | Minimal 3 batch terkirim, hasil akumulasi terbukti benar | 25% |
| E. File Sink | Streaming ke Parquet berhasil, terverifikasi dapat dibaca kembali | 20% |


## Persiapan Awal Tugas

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.functions import window, sum as spark_sum, count, col
from datetime import datetime
import os, shutil, time
import pandas as pd

spark = SparkSession.builder \
    .appName("Tugas10-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

# Bersihkan query aktif jika ada
for q in spark.streams.active:
    q.stop()

print("SparkSession Tugas 10 siap.")

SparkSession Tugas 10 siap.


## Bagian A: Menyiapkan Streaming Source (Bobot 15%)

In [3]:
folder_tugas = "tugas10_stream"
shutil.rmtree(folder_tugas, ignore_errors=True)
os.makedirs(folder_tugas, exist_ok=True)

# Definisi skema eksplisit
skema_kopi = StructType([
    StructField("pesanan_id", StringType(), True),
    StructField("nama_cabang", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("qty", IntegerType(), True),
    StructField("harga_satuan", IntegerType(), True),
    StructField("waktu_pesan", TimestampType(), True)
])

# Membuat Streaming DataFrame
df_stream = spark.readStream \
    .format("csv") \
    .schema(skema_kopi) \
    .option("header", True) \
    .load(folder_tugas)

print("Streaming source 'tugas10_stream' siap. isStreaming:", df_stream.isStreaming)

Streaming source 'tugas10_stream' siap. isStreaming: True


## Bagian B: Agregasi per Cabang (Bobot 20%)

In [4]:
# Menambahkan kolom turunan total_harga
df_dengan_total = df_stream.withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi berdasarkan nama cabang
agregasi_cabang = df_dengan_total.groupBy("nama_cabang").agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Menjalankan streaming query pertama
query_cabang = agregasi_cabang.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_cabang") \
    .start()

print("Query 1 (agregasi_cabang) berjalan.")

Query 1 (agregasi_cabang) berjalan.


## Bagian C: Agregasi Berbasis Window Waktu 2 Menit (Bobot 20%)

In [5]:
# Mendefinisikan readStream kedua dari sumber folder yang sama
df_stream_waktu = spark.readStream \
    .format("csv") \
    .schema(skema_kopi) \
    .option("header", True) \
    .load(folder_tugas) \
    .withColumn("total_harga", col("qty") * col("harga_satuan"))

# Agregasi berbasis window waktu 2 menit
agregasi_waktu = df_stream_waktu.groupBy(
    window(col("waktu_pesan"), "2 minutes")
).agg(
    spark_sum("total_harga").alias("total_pendapatan"),
    count("pesanan_id").alias("jumlah_pesanan")
)

# Menjalankan streaming query kedua
query_waktu = agregasi_waktu.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("agregasi_waktu") \
    .start()

print("Query 2 (agregasi_waktu) berjalan.")

Query 2 (agregasi_waktu) berjalan.


## Bagian D: Menguji dengan Simulasi 3 Batch Data (Bobot 25%)

In [6]:
waktu_sekarang = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# --- BATCH 1 ---
pd.DataFrame({
    "pesanan_id": ["P01", "P02"],
    "nama_cabang": ["Magelang", "Solo"],
    "nama_menu": ["Kopi Gula Aren", "Americano"],
    "qty": [2, 1],
    "harga_satuan": [20000, 18000],
    "waktu_pesan": [waktu_sekarang, waktu_sekarang]
}).to_csv(f"{folder_tugas}/batch_1.csv", index=False)

print("=== MENGIRIM BATCH 1 ===")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

# --- BATCH 2 ---
pd.DataFrame({
    "pesanan_id": ["P03", "P04"],
    "nama_cabang": ["Magelang", "Semarang"],
    "nama_menu": ["Caffe Latte", "Matcha"],
    "qty": [3, 2],
    "harga_satuan": [25000, 22000],
    "waktu_pesan": [waktu_sekarang, waktu_sekarang]
}).to_csv(f"{folder_tugas}/batch_2.csv", index=False)

print("=== MENGIRIM BATCH 2 ===")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

# --- BATCH 3 ---
pd.DataFrame({
    "pesanan_id": ["P05"],
    "nama_cabang": ["Solo"],
    "nama_menu": ["Cappuccino"],
    "qty": [2],
    "harga_satuan": [24000],
    "waktu_pesan": [waktu_sekarang]
}).to_csv(f"{folder_tugas}/batch_3.csv", index=False)

print("=== MENGIRIM BATCH 3 ===")
time.sleep(6)
spark.sql("SELECT * FROM agregasi_cabang ORDER BY total_pendapatan DESC").show()

# Cek juga hasil agregasi window waktu
print("=== HASIL WINDOW WAKTU 2 MENIT ===")
spark.sql("SELECT window.start, window.end, total_pendapatan, jumlah_pesanan FROM agregasi_waktu").show(truncate=False)

=== MENGIRIM BATCH 1 ===


+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|           40000|             1|
|       Solo|           18000|             1|
+-----------+----------------+--------------+

=== MENGIRIM BATCH 2 ===
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          115000|             2|
|   Semarang|           44000|             1|
|       Solo|           18000|             1|
+-----------+----------------+--------------+

=== MENGIRIM BATCH 3 ===
+-----------+----------------+--------------+
|nama_cabang|total_pendapatan|jumlah_pesanan|
+-----------+----------------+--------------+
|   Magelang|          115000|             2|
|       Solo|           66000|             2|
|   Semarang|           44000|             1|
+-----------+----------------+--------------+

=== HASIL WINDOW WAKTU 2 ME

## Bagian E: Sink Berbasis Berkas Parquet & Penutupan (Bobot 20%)

In [7]:
# Menyiapkan folder output & checkpoint
shutil.rmtree("tugas10_output", ignore_errors=True)
shutil.rmtree("tugas10_checkpoint", ignore_errors=True)

# Menjalankan streaming query ketiga ke format Parquet
query3 = df_stream.writeStream \
    .format("parquet") \
    .option("path", "tugas10_output") \
    .option("checkpointLocation", "tugas10_checkpoint") \
    .outputMode("append") \
    .start()

print("Query 3 (Parquet File Sink) berjalan.")

# Mengirim Batch 4 sebagai data uji penulisan file
pd.DataFrame({
    "pesanan_id": ["P06"],
    "nama_cabang": ["Semarang"],
    "nama_menu": ["Espresso"],
    "qty": [4],
    "harga_satuan": [15000],
    "waktu_pesan": [datetime.now().strftime("%Y-%m-%d %H:%M:%S")]
}).to_csv(f"{folder_tugas}/batch_4.csv", index=False)

time.sleep(6)

# 1. Membuktikan berkas Parquet tersimpan di direktori
print("\nIsi folder tugas10_output:")
print(os.listdir("tugas10_output"))

# 2. Membaca kembali file Parquet sebagai DataFrame batch biasa
print("\nMembaca hasil Parquet menggunakan spark.read.parquet():")
df_hasil_parquet = spark.read.parquet("tugas10_output")
df_hasil_parquet.show()

# 3. Menghentikan seluruh streaming query dan menutup SparkSession
query_cabang.stop()
query_waktu.stop()
query3.stop()
spark.stop()

print("Seluruh streaming query berhasil dihentikan dan SparkSession ditutup.")

Query 3 (Parquet File Sink) berjalan.



Isi folder tugas10_output:
['part-00003-67c5c6ec-e037-4f1f-ae8b-43d489608dfc-c000.snappy.parquet', '_spark_metadata', '.part-00001-22f06a29-9feb-46ff-bb6d-2826924cdc96-c000.snappy.parquet.crc', 'part-00001-22f06a29-9feb-46ff-bb6d-2826924cdc96-c000.snappy.parquet', '.part-00002-b62a94c6-1fa8-4ecb-964f-fabd70d31178-c000.snappy.parquet.crc', '.part-00003-67c5c6ec-e037-4f1f-ae8b-43d489608dfc-c000.snappy.parquet.crc', '.part-00000-b263e5d1-03a0-4372-a4e7-174c57535bc7-c000.snappy.parquet.crc', 'part-00000-b263e5d1-03a0-4372-a4e7-174c57535bc7-c000.snappy.parquet', 'part-00002-b62a94c6-1fa8-4ecb-964f-fabd70d31178-c000.snappy.parquet']

Membaca hasil Parquet menggunakan spark.read.parquet():
+----------+-----------+--------------+---+------------+-------------------+
|pesanan_id|nama_cabang|     nama_menu|qty|harga_satuan|        waktu_pesan|
+----------+-----------+--------------+---+------------+-------------------+
|       P06|   Semarang|      Espresso|  4|       15000|2026-11-13 00:32:52|